# 09 — distinct(), dropDuplicates() and sort()/orderBy()

Deduplication and ordering examples on an employees DataFrame — one operation per cell:

* `distinct()` — whole-row dedup, plus `select().distinct()` on one column or a column pair
* `dropDuplicates()` — key-based dedup on a single column (`["id"]`) and multi-column keys (`["name", "mgr_id"]`, `["mgr_id", "salary"]`)
* `dropDuplicates(subset)` accepts column-name **strings** only — `col()` objects are rejected
* `sort()` / `orderBy()` — ascending, descending, multi-column, plain-string column names
* NULL ordering — `desc_nulls_first()` / `desc_nulls_last()`


In [0]:
from pyspark.sql import (
    SparkSession,
    functions as F,
)
from pyspark.sql.functions import col, expr, lit
from pyspark.sql.types import (
    DoubleType,
    IntegerType,
    StringType,
    StructField,
    StructType,
)

spark = SparkSession.builder.appName("distinct-sorting").getOrCreate()

data = [
    (10, "Anil", 50000, 18),
    (11, "Vikas", 75000, 16),
    (12, "Nisha", 40000, 18),
    (13, "Nidhi", 60000, 17),
    (14, "Priya", 80000, 18),
    (15, "Mohit", 45000, 18),
    (16, "Rajesh", 90000, 10),
    (17, "Raman", 55000, 16),
    (18, "Sam", 65000, 17),
    (15, "Mohit", 45000, 18),
    (13, "Nidhi", 60000, 17),
    (14, "Priya", 90000, 18),
    (18, "Sam", 65000, 17),
]

schema = StructType([
    StructField("id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("salary", DoubleType(), False),
    StructField("mgr_id", IntegerType(), False),
])



schema_ddl: str = """
    id INT NOT NULL,
    name STRING,
    salary DOUBLE,
    mgr_id INT
"""

employees_df = spark.createDataFrame(data=data, schema=schema)

## The Dataset — duplicates by design

13 rows, 9 unique employees. The duplicates are deliberate:

* **Exact duplicates** — Mohit (15), Nidhi (13), Sam (18) each appear twice → removed by ANY dedup method
* **Near-duplicate** — Priya (id 14) has two rows with DIFFERENT salaries (80000 / 90000) → `distinct()` keeps BOTH (10 rows), `dropDuplicates(["id"])` keeps ONE (9 rows)

These two cases drive every count comparison in this notebook.

In [0]:
# Total Count including duplicates
print("Count:", employees_df.count())

employees_df.show()

In [0]:
# Whole-row dedup — distinct() removes exact duplicate rows
dedup_employees_df = employees_df.distinct()

print("Distinct Entire Row Count:", dedup_employees_df.count())
dedup_employees_df.show()


In [0]:
# Spark SQL Equivalent — SELECT DISTINCT removes whole-row duplicate rows
employees_df.createOrReplaceTempView("employees")

spark.sql("SELECT DISTINCT * FROM employees").show()

## Sorting — `sort()` / `orderBy()`

One sort variation per cell: ascending, descending, multi-column, plain-string column names, and the `orderBy()` alias. SQL `ORDER BY` equivalents are interleaved.

In [0]:
# Sort by salary ascending
dedup_employees_df.sort(col("salary")).show()

In [0]:
# Sort by salary in descending order
dedup_employees_df.sort(col("salary").desc()).show()

In [0]:
# Sort by salary descending, then name ascending
dedup_employees_df.sort(col("salary").desc(), col("name").asc()).show()

In [0]:
# A plain string column name is also accepted (col() preferred for chaining)
dedup_employees_df.sort("mgr_id").show()

In [0]:
# orderBy() is an alias of sort() — same result, SQL-flavoured name
dedup_employees_df.orderBy(col("salary").desc()).show()

In [0]:
# Spark SQL Equivalent — ORDER BY (ASC is the default direction)
spark.sql("""
    SELECT DISTINCT *
    FROM employees
    ORDER BY salary DESC
""").show()

## NULL Ordering — defaults and the four explicit helpers

NULL is treated as the **smallest** value: defaults put NULLs **first** on ascending, **last** on descending.
The `_nulls_first()` / `_nulls_last()` helpers override or make the default explicit. SQL: `ORDER BY ... NULLS FIRST / NULLS LAST`.

In [0]:
# Inject a NULL salary (id 15) so null-ordering behaviour is visible
employees_with_null_df = (
    employees_df
    .withColumn("salary", F.when(col("id") == 15, None).otherwise(col("salary")))
    .dropDuplicates(["id"])
)

employees_with_null_df.show()

In [0]:
# Descending salary — by default Spark puts NULLs LAST on desc (NULL is treated as the SMALLEST value)
employees_with_null_df.orderBy(col("salary").desc()).show(5)


In [0]:
# Descending salary with NULL rows pushed to the end
employees_with_null_df.sort(col("salary").desc_nulls_last()).show()

In [0]:
# Descending salary with NULL rows first
employees_with_null_df.sort(col("salary").desc_nulls_first()).show()

In [0]:
# Ascending salary — by default Spark puts NULLs FIRST on ascending (NULL is treated as the SMALLEST value)
employees_with_null_df.orderBy(col("salary").asc()).show(5)

In [0]:
# Ascending salary with NULL rows forced to the top
employees_with_null_df.sort(col("salary").asc_nulls_first()).show(5)

In [0]:
# Ascending salary with NULL rows forced to the bottom (default behaviour made explicit)
employees_with_null_df.sort(col("salary").asc_nulls_last()).show(5)

In [0]:
# Spark SQL Equivalent — DESC NULLS FIRST overrides the default (NULLs last on DESC)
employees_with_null_df.createOrReplaceTempView("employees_with_null")

spark.sql("SELECT * FROM employees_with_null ORDER BY salary DESC NULLS FIRST").show(5)

In [0]:
# Spark SQL Equivalent — ASC NULLS LAST overrides the default (NULLs first on ASC)
spark.sql("SELECT * FROM employees_with_null ORDER BY salary ASC NULLS LAST").show(5)

## `distinct()` after `select()` — dedup on a projection

`SELECT DISTINCT`-style dedup applies to the **selected columns only** — a single column or a column pair, not the whole row.

In [0]:
# distinct() after select() — unique mgr_id values only
employees_df.select(col("mgr_id")).distinct().sort(col("mgr_id")).show()

In [0]:
# distinct() on two columns — unique (mgr_id, salary) combinations
employees_df.select(col("mgr_id"), col("salary")).distinct().sort(col("mgr_id"), col("salary")).show()

In [0]:
# Spark SQL Equivalent — DISTINCT applies to the SELECTED columns (unique mgr_id, salary pairs)
spark.sql("""
    SELECT DISTINCT mgr_id, salary
    FROM employees
    ORDER BY mgr_id, salary
""").show()

## `dropDuplicates()` — key-based dedup

Whole-row by default; with a `subset` it keeps the **first row seen per key** (differences in other columns are ignored).
Covers single-column keys, multi-column keys, the snake_case `drop_duplicates()` alias, and the strings-not-`col()` subset rule.
SQL has no `DROPDUPLICATES` — `ROW_NUMBER() + rn = 1` is the equivalent pattern.

In [0]:
# dropDuplicates(subset) — keep the first row seen per id, ignore differences in other columns
employees_df.dropDuplicates(["id"]).sort(col("id")).show()

In [0]:
# drop_duplicates() — snake_case alias of dropDuplicates(); identical result and same subset rules
aliased_dedup_df = employees_df.drop_duplicates(["id"])

print("Count after drop_duplicates(['id']):", aliased_dedup_df.count())
aliased_dedup_df.sort(col("id")).show()

In [0]:
# dropDuplicates on a multi-column key — first row seen per (name, mgr_id) combination
multi_key_dedup_df = employees_df.dropDuplicates(["name", "mgr_id"])

print("Count after dropDuplicates(['name', 'mgr_id']):", multi_key_dedup_df.count())
multi_key_dedup_df.sort(col("mgr_id"), col("salary")).show()

In [0]:
# dropDuplicates on (mgr_id, salary) — first row seen per manager-salary combination
mgr_salary_dedup_df = employees_df.dropDuplicates(["mgr_id", "salary"])

print("Count after dropDuplicates(['mgr_id', 'salary']):", mgr_salary_dedup_df.count())
mgr_salary_dedup_df.sort(col("mgr_id"), col("salary")).show()

In [0]:
# dropDuplicates(subset) accepts column NAMES as strings only — Column objects from col() are rejected
try:
    employees_df.dropDuplicates([col("mgr_id"), col("salary")]).show()
except Exception as e:
    print("Error: dropDuplicates() 'subset' must be a list of column-name STRINGS,")
    print("e.g. dropDuplicates(['mgr_id', 'salary']) — Column objects from col() are NOT accepted.")

In [0]:
# distinct() keeps both Priya rows (different salaries); dropDuplicates(['id']) keeps one
print("distinct()            :", employees_df.distinct().count())
print("dropDuplicates(['id']) :", employees_df.dropDuplicates(["id"]).count())

In [0]:
# Spark SQL Equivalent of dropDuplicates(['id']) — ROW_NUMBER() keeps ONE row per id key
# SQL has no dropDuplicates; ROW_NUMBER + rn = 1 is the standard pattern (and it's DETERMINISTIC —
# ORDER BY salary DESC always keeps Priya's 90000 row, unlike dropDuplicates' first-row-seen)
spark.sql("""
    SELECT id, name, salary, mgr_id
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY id ORDER BY salary DESC) AS rn
        FROM employees
    )
    WHERE rn = 1
    ORDER BY id
""").show()

## Exercise — Find Customer Referee

**LeetCode:** [Find Customer Referee — LeetCode 584](https://leetcode.com/problems/find-customer-referee/description/?envType=study-plan-v2&envId=top-sql-50) (part of the *Top SQL 50* study plan)

Solve this yourself using **both** the DataFrame API and Spark SQL, with what you've learned so far —
`select()`, `filter()/where()`, `col()` expressions, and NULL handling.

**Table: Customer**

```
+-------------+---------+
| Column Name | Type    |
+-------------+---------+
| id          | int     |
| name        | varchar |
| referee_id  | int     |
+-------------+---------+
```

In SQL, `id` is the primary key column for this table.
Each row indicates the id of a customer, their name, and the id of the customer who referred them.

**Problem — find the names of the customers that are either:**

* referred by any customer with `id != 2`
* not referred by any customer (`referee_id` is NULL)

Return the result table in any order.

**Example 1**

Input:

```
Customer table:
+----+------+------------+
| id | name | referee_id |
+----+------+------------+
| 1  | Will | null       |
| 2  | Jane | null       |
| 3  | Alex | 2          |
| 4  | Bill | null       |
| 5  | Zack | 1          |
| 6  | Mark | 2          |
+----+------+------------+
```

Output:

```
+------+
| name |
+------+
| Will |
| Jane |
| Bill |
| Zack |
+------+
```

> [!TIP]
> 💡 Hint — before writing the filter, think carefully about what `referee_id != 2` does to the NULL rows.

In [0]:
# Create a customers DataFrame using the problem statement example
# NULL referee_id = not referred by anyone

customer_data = [
    (1, "Will", None),
    (2, "Jane", None),
    (3, "Alex", 2),
    (4, "Bill", None),
    (5, "Zack", 1),
    (6, "Mark", 2),
]

customer_df = spark.createDataFrame(customer_data, schema="id INT, name STRING, referee_id INT")

customer_df.show()


### Solution - DataFrame API

In [0]:
df_result = (
    customer_df
    .filter( 
        col("referee_id").isNull() | (col("referee_id") != lit(2))
    )
    .select(col("name"))
    .sort(col("name")) # OR .orderBy(col("name"))
)

df_result.show()


### Solution — Spark SQL

In [0]:
customer_df.createOrReplaceTempView("customer")

df_result = spark.sql("""
    SELECT name
    FROM customer
    WHERE referee_id IS NULL
        OR referee_id != 2
    ORDER BY name
""")

df_result.show()

## Column References — plain string vs `col()`

Both styles are accepted when specifying columns:

| Style | Example | Works? |
| --- | --- | --- |
| Plain string | `df.sort("salary")` | ✅ Yes |
| `col()` expression | `df.sort(col("salary"))` | ✅ Yes |

```python
employees_df.sort("salary")            # string column name — fine
employees_df.sort(col("salary"))        # Column expression — fine
```

> [!TIP]
> 💡 Stick to **one style consistently — `col()` is preferred**. A `col()` Column object lets you chain transformations directly:

```python
df.sort(col("salary").desc())
df.sort(col("salary").desc_nulls_last())
df.select(col("salary").alias("pay"))
df.filter(col("salary") > 50000)
df.withColumn("pay_k", col("salary") / 1000)
```

> [!WARNING]
> ⚠️ With a plain string you CANNOT chain `desc()`, `desc_nulls_last()`, `alias()`, `cast()`, etc. — those methods live on the `Column` object returned by `col()`, not on `str`. Mixing both styles in the same notebook also makes code harder to read.

## Quick Revision — `distinct()` vs `dropDuplicates()` vs `sort()`

| Method | Dedup / order scope | Notes |
| --- | --- | --- |
| `df.distinct()` | ALL columns (whole row) | Equivalent to `SELECT DISTINCT *` |
| `df.dropDuplicates()` | ALL columns | Same result as `distinct()` |
| `df.drop_duplicates()` | ALL columns / subset | snake_case **alias** of `dropDuplicates()` — identical behaviour |
| `df.dropDuplicates(["id"])` | Only listed columns | 💡 Keeps the **first** row seen per key — other columns ignored |
| `df.dropDuplicates([col("id")])` ❌  `df.dropDuplicates(["id"])` ✅ | — | ❌ NOT accepted — `subset` takes column-name **strings** only, not `Column` objects |
| `df.select("mgr_id").distinct()` | Only selected columns | Dedup after projection |
| `df.sort(col(...))` / `df.orderBy(col(...))` | — | `orderBy()` is an alias of `sort()` |
| `col("salary").desc_nulls_last()` | — | `NULL`s sort last in descending — the DEFAULT; use `desc_nulls_first()` to override |

> [!TIP]
> 💡 Use `dropDuplicates(subset)` when duplicates differ in only a few columns and you want the first occurrence per key.

> [!WARNING]
> ⚠️ `sort()` with NULLs — Spark treats `NULL` as the SMALLEST value: NULLs appear **first** on ascending, **last** on descending. Override with `asc_nulls_last()` / `desc_nulls_first()`.

```python
dedup_df = employees_df.dropDuplicates(["id"]).sort(col("salary").desc(), col("name"))

df.dropDuplicates(["id", "mgr_id"])   # ✅ subset = list of column-name STRINGS
df.dropDuplicates([col("id")])        # ❌ NOT_EXPECTED_TYPE — col() objects rejected
```

# Quick Revision Notes (Handwritten-Style) — DISTINCT, dropDuplicates & SORT

## 1 — Syntax map

* Whole-row dedup: `df.distinct()` == `df.dropDuplicates()` == SQL `SELECT DISTINCT *`
* Key-based dedup: `df.dropDuplicates(["id"])` — first row per key
* Alias: `df.drop_duplicates(["id"])` — snake_case alias of `dropDuplicates()`, identical behaviour
* Sort: `df.sort(col("salary").desc())` == `df.orderBy(...)` — alias, same thing

## 2 — distinct() vs dropDuplicates()

* `distinct()` → ALL columns (whole row); `dropDuplicates()` no args → same result
* `dropDuplicates(subset)` → dedup ONLY on listed key columns
  💡 keeps the **FIRST** row seen per key — other columns ignored
* select().distinct() → dedup AFTER projection (one column or a pair)

## 3 — The Priya example (counts to remember)

* Priya (id 14) appears twice with DIFFERENT salaries (80000 / 90000)
* `distinct()` keeps BOTH → 10 rows
* `dropDuplicates(["id"])` keeps ONE → 9 rows
* `dropDuplicates(["name", "mgr_id"])` → 9 (same key, collapses)
* `dropDuplicates(["mgr_id", "salary"])` → 10 (both keys unique, both survive)
* ⚠️ The KEY choice decides which rows collapse — not the method

## 4 — Sorting

* Single: `sort(col("salary"))` / `.desc()`
* Multi-column: `sort(col("salary").desc(), col("name").asc())` — leftmost column first
* Plain string OK: `sort("salary")` — but no chaining, prefer `col()`

## 5 — NULL ordering ⚠️

* Spark treats NULL as the SMALLEST value
* ASC default → NULLs FIRST; DESC default → NULLs LAST
* Override: `asc_nulls_last()`, `desc_nulls_first()`
* `desc_nulls_last()` / `asc_nulls_first()` just make the DEFAULT explicit

## 6 — dropDuplicates() subset rule

* subset = list of column-name **STRINGS** only
* `col()` objects → NOT_EXPECTED_TYPE error ❌

## 7 — Column reference styles

* String `"salary"` and `col("salary")` both work
* 💡 Stick to `col()` — chains `desc()`, `desc_nulls_last()`, `alias()`, `cast()`

## Method cheat-sheet

| Method | Scope | Keep this in mind |
| --- | --- | --- |
| `distinct()` | whole row | = `SELECT DISTINCT *` |
| `dropDuplicates()` | whole row | same as distinct() |
| `dropDuplicates(["id"])` | key only | first row per key |
| `select("mgr_id").distinct()` | projected | dedup after select |
| `sort()` / `orderBy()` | order | alias pair; NULL = smallest |

## Quick Recall Box ⭐

* `distinct()` == `dropDuplicates()` (no args) — whole row
* `drop_duplicates()` = snake_case alias of `dropDuplicates()` (both exist in PySpark)
* `dropDuplicates(subset)` keeps FIRST row per key — other columns ignored
* subset takes STRINGS, never `col()` objects
* `orderBy()` is just an alias of `sort()`
* NULL is the SMALLEST value → asc puts NULLs first, desc puts them last
* `desc_nulls_last()` is the desc DEFAULT; `desc_nulls_first()` overrides it
* Priya test: distinct = 10, dropDuplicates(["id"]) = 9 — remember why!
* Key includes the differing column → rows survive (mgr_id+salary)
* Prefer `col()` over plain strings for chaining